# Vaizdų generavimo programų kūrimas (Azure OpenAI)

LLM (didelės kalbos modeliai) nesudaromi tik tekstui generuoti. Jūs taip pat galite generuoti vaizdus iš teksto aprašymų. Vaizdai kaip modulis yra naudingi MedTech, architektūros, turizmo, žaidimų kūrimo, rinkodaros ir kitose srityse. Šioje pamokoje dirbsime su šiandieniniais **GPT Image** modeliais Microsoft Foundry aplinkoje.

## Mokymosi tikslai

Šios pamokos pabaigoje gebėsite:

- Paaiškinti, kas yra vaizdų generavimas ir kur jis naudingas.
- Suprasti `gpt-image` modelių šeimą ir kuo ji skiriasi nuo senųjų DALL·E modelių.
- Kurti vaizdų generavimo programą ir redaguoti vaizdus.

## Kas yra vaizdų generavimas?

Vaizdų generavimo modeliai kuria vaizdus pagal teksto užklausą. Šiuolaikiniai modeliai, tokie kaip `gpt-image`, mokosi teksto ir vaizdų tarpusavio ryšio treniruočių metu, o vėliau iteratyviai paverčia atsitiktinį triukšmą į vaizdą, atitinkantį jūsų užklausą.

Dvi gerai žinomos vaizdų modelių šeimos yra:

- **`gpt-image` (OpenAI)** — dabartinė kartos šeima, naudojama šioje pamokoje. Ji palaiko teksto į vaizdą generavimą ir vaizdų redagavimą (įpaišymą su kauke).
- **Midjourney** — populiarus trečiųjų šalių modelis su savo paslauga ir darbų eiga Discord platformoje.

> Senesni OpenAI vaizdų modeliai — **DALL·E 2** ir **DALL·E 3** — yra pasenę. DALL·E 3 nebėra prieinamas naujiems diegimams, o funkcijos kaip `create_variation` egzistavo tik DALL·E 2. Naujiems projektams naudokite `gpt-image` modelius.

Microsoft Foundry aplinkoje, **`gpt-image-2`** yra naujausias ir pajėgiausias vaizdų modelis ir yra rekomenduojamas kaip numatytasis. Taip pat yra prieinami `gpt-image-1.5` ir `gpt-image-1-mini`.

> **Svarbu:** `gpt-image` modeliai grąžina sugeneruotą vaizdą kaip **base64** (`b64_json`), o ne kaip URL. Jūsų kodas dekoduoja base64 eilutę į baitus ir juos išsaugo — nėra vaizdo URL, iš kurio būtų galima jį atsisiųsti.


## Kaip sukurti pirmąją paveikslėlių generavimo programą

Tai ką reikia norint sukurti paveikslėlių generavimo programą? Jums reikės šių bibliotekų:

- **python-dotenv**, labai rekomenduojama naudoti šią biblioteką, kad paslėptumėte savo paslaptis *.env* faile nuo kodo.
- **openai**, ši biblioteka bus naudojama bendraujant su OpenAI API.
- **pillow**, dirbti su paveikslėliais Python kalba.

Jei dar nepadaryta, vykdykite nurodymus [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) puslapyje, kad sukurtumėte Microsoft Foundry išteklių ir modelį. Pasirinkite **gpt-image-2** modelį (naujausias Azure OpenAI paveikslėlių modelis; DALL·E yra senstelėjęs).

1. Sukurkite *.env* failą su šiuo turiniu:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Šią informaciją surasite Microsoft Foundry portale „Deployments“ skyriuje prie savo ištekliaus.



1. Surinkite aukščiau išvardytas bibliotekas faile, pavadintame *requirements.txt*, taip:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Toliau sukurkite virtualią aplinką ir įdiekite bibliotekas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!PASTABA]
> Windows vartotojams naudokite šias komandas virtualiai aplinkai sukurti ir aktyvuoti:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Įdėkite šį kodą į failą pavadinimu *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # importuoti dotenv
    dotenv.load_dotenv()

    # sukonfigūruoti Azure OpenAI (Microsoft Foundry) klientą.
    # Vaizdo modeliams reikalinga naujesnė API versija – patikrinkite Foundry dokumentaciją, kuri versija reikalinga jūsų modeliui.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Sukurti vaizdą naudodami vaizdų generavimo API
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Įveskite savo užklausos tekstą čia
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # pvz. "gpt-image-2"
        )
        # Nustatyti katalogą saugomam vaizdui
        image_dir = os.path.join(os.curdir, 'images')

        # Jei katalogas neegzistuoja, sukurkite jį
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializuoti vaizdo kelią (atkreipkite dėmesį, kad failo tipas turi būti png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modeliai grąžina vaizdą kaip base64 (b64_json), o ne URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Rodyti vaizdą numatytoje vaizdo peržiūros programoje
        image = Image.open(image_path)
        image.show()

    # sugauti išimtis
    except BadRequestError as err:
        print(err)

    ```

Paaiškinkime šį kodą:

- Pirmiausia importuojame reikalingas bibliotekas, įskaitant OpenAI biblioteką, dotenv biblioteką, base64 modulį ir Pillow biblioteką.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Tada užkrauname aplinkos kintamuosius iš *.env* failo.

    ```python
    # importuoti dotenv
    dotenv.load_dotenv()
    ```

- Po to konfigūruojame Azure OpenAI (Microsoft Foundry) klientą.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Tada generuojame paveikslėlį:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Įveskite savo užklausos tekstą čia
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` modeliai grąžina paveikslėlį kaip **base64** eilutę `data[0].b64_json`. Mes ją dekoduojame į baitus ir išsaugome į failą — nėra URL atsisiuntimui.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Galiausiai atidarome paveikslėlį ir jį parodome naudodami standartinį vaizdų peržiūros įrankį:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Daugiau detalių apie paveikslėlio generavimą

Pažvelkime į `images.generate` parametrus:

- **prompt**, tai teksto užklausa, naudojama paveikslėliui generuoti. Čia ji „Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils“.
- **size**, tai generuojamo paveikslėlio dydis (pavyzdžiui `1024x1024`, `1536x1024`, `1024x1536` arba `"auto"`).
- **n**, tai generuojamų paveikslėlių skaičius. Čia generuojame vieną.
- **model**, tai jūsų paveikslėlio modelio įdiegimo pavadinimas (pavyzdžiui `gpt-image-2`).

> Paveikslėlių modeliai negali priimti `temperature` parametro — tai yra teksto generavimo kontrolė. Kad gautumėte įvairovės, iškvieskite API dar kartą; norėdami sumažinti įvairovę, padarykite užklausą tikslesnę.

## Papildomos galimybės generuojant paveikslėlius

Jūs jau matėte, kaip su keletu Python eilučių sugeneruoti paveikslėlį. `gpt-image` modeliai taip pat gali **redaguoti** esamą paveikslėlį. Pateikdami paveikslėlį, pasirenkamą **kaukę** (kuria nurodomas keičiamas plotas) ir užklausą, galite pakeisti dalį paveikslėlio — pavyzdžiui, pridėti kepurę mūsų kiškiui.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# redagavimai taip pat grąžinami kaip base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Pradinėje nuotraukoje matome tik triušį; galutinėje nuotraukoje triušiui uždėta kepurė.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
